# Hybrid surrogate — t=100 run (NOTEBOOK v1-t100, 2026-07-04)

Trains `configs/hybrid_sw_gate_s1em3_t100.yaml`: identical model/optimiser/seed to the
published t=50 run, evolution extended to **t=100 M** (motivated by the end-time scan:
fine-field tau at xq=10 improves 2.6% -> 0.8% as t_end extends to ~80 M).

**Runtime: the 96 GB GPU (H100-class).** Grid is 2x the t=50 run; peak ~76 GB at batch 8.

**Rules (learned the hard way):**
1. **NEVER use File → Save a copy in GitHub** — it overwrites the repo.
2. Run cells top to bottom in one session. If the runtime dies, re-run from Cell 2.
3. Corpus build (Cell 4) is CPU-serial: expect **~1.5–3 h**. Training (Cell 5): **~2–2.5 h**.
4. Download the artifact zip (Cell 7) BEFORE closing the tab.

In [ ]:
# Cell 2 — sanity: GPU, CPU count, RAM
!nvidia-smi
!nproc
!free -g | head -2

In [ ]:
# Cell 3 — clone + pinned deps + allocator flag (set BEFORE any torch import)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

%cd /content
!rm -rf QNM_Project_Fresh
!git clone --depth 1 https://github.com/ScreaM835/QNM_Project_Fresh.git
%cd /content/QNM_Project_Fresh
!pip -q install "neuraloperator==2.0.0"
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0))
print('ALLOC_CONF =', os.environ['PYTORCH_CUDA_ALLOC_CONF'])

In [ ]:
# Cell 4 — build the t=100 corpora (k=4 prior + k=2 Richardson label).
# Serial FD solves: ~1.5-3 h total. Same Sobol seed as t=50 => identical 1200 draws.
%cd /content/QNM_Project_Fresh
!python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset_t100.yaml \
    --k 4 --out outputs/hybrid/dataset_sw_k4_t100.npz
!python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset_t100.yaml \
    --k 2 --out outputs/hybrid/dataset_sw_k2_t100.npz
!ls -lh outputs/hybrid/*.npz

In [ ]:
# Cell 5 — train (~2-2.5 h on the 96 GB card at batch 8)
%cd /content/QNM_Project_Fresh
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python scripts/train_hybrid_fno.py --config configs/hybrid_sw_gate_s1em3_t100.yaml

In [ ]:
# Cell 6 — quick training summary
import json
with open('/content/QNM_Project_Fresh/outputs/hybrid/fno_sw_gate_s1em3_t100/history.json') as f:
    h = json.load(f)
hist = h['history']
print('epochs        :', len(hist))
print('best val MSE  :', h['best_val_mse'])
print('first/last val:', hist[0]['val_mse'], '/', hist[-1]['val_mse'])
print('(t=50 reference: best val 3.878e-7 — not directly comparable; different target field)')

In [ ]:
# Cell 7 — zip + download artifacts (model, history, checkpoint)
%cd /content/QNM_Project_Fresh
!zip -r /content/hybrid_t100_results.zip \
    outputs/hybrid/fno_sw_gate_s1em3_t100 \
    -x 'outputs/hybrid/fno_sw_gate_s1em3_t100/field_cache/*'
from google.colab import files
files.download('/content/hybrid_t100_results.zip')

In [ ]:
# Cell 8 — QNM eval: t_end scan on the trained t=100 model (same session, corpus in place).
# Canonical [10,50] plus extended windows 65/80/100 to trace where tau saturates.
# Pulls the latest (parametrised) eval script without touching the built corpus/model.
%cd /content/QNM_Project_Fresh
!git stash -u 2>/dev/null; git pull --no-edit
for TE in 50 65 80 100:
    print(f'\n================  t_end = {TE}  ================')
    !python scripts/eval_hybrid_protocol1.py \
        --config configs/hybrid_sw_gate_s1em3_t100.yaml --t_end {TE} --procs 8

In [ ]:
# Cell 9 — zip + download the eval JSONs (small; keep even if the runtime dies)
%cd /content/QNM_Project_Fresh
!zip -j /content/hybrid_t100_eval.zip \
    outputs/hybrid/fno_sw_gate_s1em3_t100/eval/protocol1_xq10_te*.json
from google.colab import files
files.download('/content/hybrid_t100_eval.zip')